In [149]:
import pandas as pd
df=pd.read_csv('appearances.csv')

In [150]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1706806 entries, 0 to 1706805
Data columns (total 13 columns):
 #   Column                  Dtype 
---  ------                  ----- 
 0   appearance_id           object
 1   game_id                 int64 
 2   player_id               int64 
 3   player_club_id          int64 
 4   player_current_club_id  int64 
 5   date                    object
 6   player_name             object
 7   competition_id          object
 8   yellow_cards            int64 
 9   red_cards               int64 
 10  goals                   int64 
 11  assists                 int64 
 12  minutes_played          int64 
dtypes: int64(9), object(4)
memory usage: 169.3+ MB


In [151]:
df.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,6646,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


    Dropping unnecessary columns that have no predictive features

In [152]:
df.drop(columns=['player_name', 'appearance_id', 'game_id', 'player_current_club_id', 'player_club_id'],axis=1,inplace=True)

In [153]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1706806 entries, 0 to 1706805
Data columns (total 8 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   player_id       int64 
 1   date            object
 2   competition_id  object
 3   yellow_cards    int64 
 4   red_cards       int64 
 5   goals           int64 
 6   assists         int64 
 7   minutes_played  int64 
dtypes: int64(6), object(2)
memory usage: 104.2+ MB


    Handling missing values

In [154]:
df.isnull().sum()

player_id         0
date              0
competition_id    0
yellow_cards      0
red_cards         0
goals             0
assists           0
minutes_played    0
dtype: int64

    Encoding

In [155]:
from sklearn.preprocessing import LabelEncoder

def encoding(df):
    encoder=LabelEncoder()
    for col in df.columns:
        if df[col].dtype=='object':
            if df[col].nunique()<5:
                dummies=pd.get_dummies(df[col],prefix=col,dtype=int)
                df=pd.concat([df.drop(columns=[col]),dummies],axis=1)
            else:
                df[col]=encoder.fit_transform(df[col])
    return df


In [156]:
df=encoding(df)

In [157]:
df.head()

,player_id,date,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,38004,0,6,0,0,2,0,90
1,79232,1,13,0,0,0,0,90
2,42792,1,13,0,0,0,0,45
3,73333,1,13,0,0,0,0,90
4,122011,1,13,0,0,0,1,90


In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1706806 entries, 0 to 1706805
Data columns (total 8 columns):
 #   Column          Dtype
---  ------          -----
 0   player_id       int64
 1   date            int32
 2   competition_id  int32
 3   yellow_cards    int64
 4   red_cards       int64
 5   goals           int64
 6   assists         int64
 7   minutes_played  int64
dtypes: int32(2), int64(6)
memory usage: 91.2 MB


In [159]:
df.head()

,player_id,date,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,38004,0,6,0,0,2,0,90
1,79232,1,13,0,0,0,0,90
2,42792,1,13,0,0,0,0,45
3,73333,1,13,0,0,0,0,90
4,122011,1,13,0,0,0,1,90


    Scaling

In [160]:

from sklearn.preprocessing import MinMaxScaler
def scaling_qil(df):
    scaler=MinMaxScaler()
    num_col=df.select_dtypes(include=['int32','int64']).columns.drop('goals')
    df[num_col]=scaler.fit_transform(df[num_col])
    return df


In [161]:
df=scaling_qil(df)

In [170]:
df.head()

,player_id,date,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,0.027515,0.000000,0.142857,0.0,0.0,2,0.000000,0.605442
1,0.057371,0.000268,0.309524,0.0,0.0,0,0.000000,0.605442
2,0.030982,0.000268,0.309524,0.0,0.0,0,0.000000,0.299320
3,0.053099,0.000268,0.309524,0.0,0.0,0,0.000000,0.605442
4,0.088351,0.000268,0.309524,0.0,0.0,0,0.166667,0.605442


# Training

In [171]:
x=df.drop('goals',axis=1)

In [172]:
y=df['goals']

In [173]:
from sklearn.model_selection import  train_test_split

In [174]:
x_train, x_test,y_train, y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [175]:
df['goals'].value_counts()

goals
0    1560446
1     130986
2      13631
3       1572
4        147
5         23
6          1
Name: count, dtype: int64

In [178]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()

In [179]:
lr_model.fit(x_train, y_train)

LinearRegression()

In [180]:
y_pred = lr_model.predict(x_test)

In [181]:
y_pred

array([0.08338929, 0.11295903, 0.18921433, ..., 0.13873124, 0.1087917 ,
       0.10194529])

In [183]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae, mse, rmse, r2)

0.1736006432613083 0.1091378291978992 0.33036015074142827 0.011059912895311497


    Overall the errors are pretty high and r2 score is low. this means that model performance is not good enough.